# Offline ALNS Repair Model Training — Improved Pipeline

**Colab-ready** | Explicit seeds | Dataset integrity checks | Quality gates | Covariate-shift mitigation

This notebook runs the full offline training pipeline for the Hybrid ALNS repair model.
It is the authoritative, executable version of the improvements documented in `README.md`.

## Pipeline overview

```
generate_dataset (v1) ──► train baseline (v1) ──► collect ALNS states
                                                          │
                         generate_dataset (v2) ──────────┘
                                  │
                         train improved model (v2)  ← quality gate checked here
```

## Improvement areas addressed

| Area | What changed |
|---|---|
| **Reproducibility** | All seeds defined once as constants; passed to every step |
| **Dataset integrity** | Feature version, feature count, label values, NaN checked on load |
| **Dataset summary** | Rows, positive rate, per-source counts, ALNS ratio printed |
| **Quality gates** | ROC-AUC + Average Precision checked on holdout; warns if below threshold |
| **Covariate shift** | ALNS states required for v2+ (`require_alns_states=True`); proportion reported |


## Cell 1 — Colab / local setup

Detects whether the notebook is running on Google Colab or locally.
On Colab, it prompts you to upload the repository zip and installs it.
Locally, it walks up from the current directory to find the repo root.

> **Colab tip**: if this is your first run, upload `bin-packing-optimization.zip`
> when the file picker appears. Subsequent runs can skip the upload if the
> `/content` directory still has the extracted repo.


In [10]:
import os
import sys
import shutil
from pathlib import Path

# Where we WANT the repository files to live
repo_root = Path("/kaggle/working/bin-packing-optimization")

# Precise path where Kaggle mounted your uploaded files
kaggle_input_dir = Path("/kaggle/input/datasets/ademabdelhafidhdiar/bin-packing-optimization")

print("Initializing writable workspace...")
if repo_root.exists():
    shutil.rmtree(repo_root)

# Custom copy function that ignores files containing bracket formatting characters
def copy_clean_tree(src, dst):
    os.makedirs(dst, exist_ok=True)
    for item in os.listdir(src):
        s = os.path.join(src, item)
        d = os.path.join(dst, item)
        
        # Skip files with forbidden Kaggle validation characters like '[' or ']'
        if '[' in item or ']' in item:
            print(f"Skipping forbidden documentation file: {item}")
            continue
            
        if os.path.isdir(s):
            copy_clean_tree(s, d)
        else:
            shutil.copy2(s, d)

print("Copying repository files into writable workspace...")
copy_clean_tree(kaggle_input_dir, repo_root)

# --- NESTING FIX STAGE ---
# Check if there's a double nested 'bin-packing-optimization' folder inside repo_root
nested_folder_v1 = repo_root / "bin-packing-optimization"
nested_folder_v2 = repo_root / "bin_packing_optimization"
target_nested = nested_folder_v1 if nested_folder_v1.exists() else (nested_folder_v2 if nested_folder_v2.exists() else None)

# If we find a setup.py inside a subfolder, move everything up one level to flatten it
if target_nested and (target_nested / "requirements.txt").exists():
    print(f"Detected nested directory layout at {target_nested}. Flattening directory structure...")
    temp_dir = Path("/kaggle/working/temp_flatten")
    
    # Move files out to a temp spot, clear the root, and bring them back flat
    shutil.move(str(target_nested), str(temp_dir))
    shutil.rmtree(repo_root)
    shutil.move(str(temp_dir), str(repo_root))

# Register the clean repository path so Python absolute imports work seamlessly
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

os.chdir(repo_root)
print(f"\n✓ Success! Active directory set to: {os.getcwd()}")
print("Contents of active directory:", os.listdir(os.getcwd()))

Initializing writable workspace...
Copying repository files into writable workspace...
Detected nested directory layout at /kaggle/working/bin-packing-optimization/bin_packing_optimization. Flattening directory structure...

✓ Success! Active directory set to: /kaggle/working/bin-packing-optimization
Contents of active directory: ['pyproject.toml', 'requirements.txt', 'LICENSE', 'bin_packing_optimization.egg-info', 'bin_packing_optimization', 'README.md']


## Cell 2 — Install dependencies

Installs Python dependencies from `requirements.txt` and registers the repo
as an editable package so all internal imports resolve correctly.
The `-q` flag suppresses verbose pip output to keep the notebook readable.


In [11]:
!pip install -q -r requirements.txt
!pip install -q -e .


  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for bin-packing-optimization (pyproject.toml) ... done


## Cell 3 — Seed constants and path setup

All randomness is controlled by four constants defined here.
Using separate seeds per step allows independent auditing and partial reruns
without affecting other steps.

| Constant | Used by | Purpose |
|---|---|---|
| `SEED` | `train_repair_model` | Train/test split + GradientBoosting `random_state` |
| `SYNTHETIC_V1_SEED` | `generate_dataset` (v1) | Baseline synthetic dataset |
| `ALNS_SEED` | `collect_alns_states` | ALNS rollout RNG |
| `SYNTHETIC_V2_SEED` | `generate_dataset` (v2) | Supplementary synthetic dataset |


In [12]:
import random
import numpy as np
import sys
import os
from pathlib import Path

# Reproducibility configurations
SEED              = 42   
SYNTHETIC_V1_SEED = 0    
ALNS_SEED         = 1    
SYNTHETIC_V2_SEED = 2    

random.seed(SEED)
np.random.seed(SEED)

# Reroute internal pipeline targets directly into the writable directory tree
TRAINING_DIR = (
    Path("/kaggle/working/bin-packing-optimization") 
    / "bin_packing_optimization" 
    / "hybrid_learning_metaheuristics" 
    / "hybrid_alns" 
    / "repair_model_training"
)
DATA_DIR = TRAINING_DIR / "training_data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(TRAINING_DIR)
print("Pipeline targets established at:", TRAINING_DIR)

# Core package execution imports
from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns.repair_model_training.generate_dataset import (
    GenerateDatasetConfig, generate_dataset,
)
from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns.repair_model_training.collect_alns_states import (
    CollectAlnsStatesConfig, collect_alns_states,
)
from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns.repair_model_training.train_repair_model import (
    TrainRepairModelConfig, train_repair_model,
)

Pipeline targets established at: /kaggle/working/bin-packing-optimization/bin_packing_optimization/hybrid_learning_metaheuristics/hybrid_alns/repair_model_training


## Step 1 — Generate baseline synthetic dataset

Generates `synthetic_v1.pkl`: 4 000 random bin-packing instances with items drawn
from uniform, bimodal, and Gaussian distributions.

For each instance the FFD heuristic builds a start solution; then every feasible
bin placement is labelled using the BFD oracle (1 = minimum-slack bin, 0 = other).

**Expected output** (printed by `generate_dataset`):
- Total rows, positive rate (~0.17 for `max_negatives=5`), class counts.
- An integrity check is run automatically before saving.


In [13]:
generate_dataset(
    GenerateDatasetConfig(
        instances=4000,
        n_min=50,
        n_max=200,
        max_negatives=5,
        seed=SYNTHETIC_V1_SEED,
        workers=1,
        output=str(DATA_DIR / "synthetic_v1.pkl"),
    )
)

# Persistent backup step
!cp {str(DATA_DIR / "synthetic_v1.pkl")} /kaggle/working/synthetic_v1.pkl
print("✓ Dataset v1 backed up safely to persistent output root.")

DATASET GENERATION
  Instances          : 4000
  Instance size range: [50, 200] items
  Max negatives      : 5
  Seed               : 0
  Workers            : 1
  Output             : /kaggle/working/bin-packing-optimization/bin_packing_optimization/hybrid_learning_metaheuristics/hybrid_alns/repair_model_training/training_data/synthetic_v1.pkl


Generating dataset: 100%|██████████| 4000/4000 [00:09<00:00, 442.52it/s]



Dataset: 707,772 rows x 11 features
  Positive rate : 0.3224  (expected min: 0.1667)
  Class 0 (neg) : 479,551
  Class 1 (pos) : 228,221

Saved: /kaggle/working/bin-packing-optimization/bin_packing_optimization/hybrid_learning_metaheuristics/hybrid_alns/repair_model_training/training_data/synthetic_v1.pkl  (32.4 MB)
Next step: pass this file to train_repair_model.py via the data config field
✓ Dataset v1 backed up safely to persistent output root.


## Step 2 — Train baseline model (v1)

Trains a `GradientBoostingClassifier` on `synthetic_v1.pkl` only.
This is the v1 baseline — no ALNS states are required yet
(`require_alns_states=False`).

**What this cell does:**
- Prints seed provenance for the train/test split and model `random_state`.
- Runs integrity checks on the dataset (feature version, NaN, label values).
- Prints a dataset summary: rows, positive rate, per-source counts.
- Trains with class-balanced sample weights and early stopping.
- Evaluates ROC-AUC + Average Precision on a 15% holdout.
- Checks quality gates and warns if scores are below the thresholds.
- Saves `repair_model_v1.pkl` with the model, scaler, metrics, and seed provenance.

**Quality thresholds**: ROC-AUC ≥ 0.80, Average Precision ≥ 0.60.


In [14]:
train_repair_model(
    TrainRepairModelConfig(
        data=[str(DATA_DIR / "synthetic_v1.pkl")],
        output=str(TRAINING_DIR / "repair_model_v1.pkl"),
        seed=SEED,
        min_roc_auc=0.80,
        min_average_precision=0.60,
        require_alns_states=False,
        cv_folds=5,
        no_learning_curves=True,
        no_plots=True,
    )
)

# Persistent backup step
!cp {str(TRAINING_DIR / "repair_model_v1.pkl")} /kaggle/working/repair_model_v1.pkl
print("✓ Model v1 weights backed up safely to persistent output root.")

REPRODUCIBILITY — SEEDS
  config.seed          : 42  (train/test split + model)
  Python random seed   : set to 42
  numpy random seed    : set to 42

PHASE 1: LOADING DATASET(S)

────────────────────────────────────────────────────────────
DATASET INTEGRITY CHECKS
────────────────────────────────────────────────────────────
  ✓ synthetic_v1.pkl
      source       : synthetic
      rows         : 707,772
      feature_ver  : 2
      pos_rate     : 0.3224
      NaN/inf      : none
      labels       : [0, 1]

  Merged totals
    rows         : 707,772
    positive rate: 0.3224
    ALNS rows    : 0  (0.0% of total)
    [synthetic] : 707,772 rows
────────────────────────────────────────────────────────────

Merged dataset : 707,772 rows x 11 features
  Positive rate : 0.3224
  Class 0 (neg) : 479,551
  Class 1 (pos) : 228,221
  ALNS ratio    : 0.0%

PHASE 2: TRAIN/TEST SPLIT
  random_state = 42
Training set : 601,606 samples
Test set     : 106,166 samples

PHASE 3: MODEL TRAINING
Training

## Step 3 — Collect ALNS repair states (covariate-shift mitigation)

The baseline model is trained on BFD-labelled FFD states. During actual ALNS
search, the solver visits states that BFD never produces — partially destroyed
solutions with arbitrary residual bin loads. This **covariate shift** can degrade
repair quality.

`collect_alns_states` runs the v1 model on 500 fresh instances, captures every
repair state it encounters, and re-labels those states with the BFD oracle.
This is a single-pass **DAgger-lite** approach: cheaper than true DAgger, but
already substantially reduces the distribution gap.

**Expected output**: row count and positive rate for the captured states;
the file is tagged `source='alns_states'` so the training script can reliably
count and report the ALNS proportion.


In [15]:
collect_alns_states(
    CollectAlnsStatesConfig(
        model_path=str(TRAINING_DIR / "repair_model_v1.pkl"),
        instances=500,
        n_min=50,
        n_max=200,
        max_negatives=5,
        iterations=200,
        seed=ALNS_SEED,
        output=str(DATA_DIR / "alns_states_v1.pkl"),
    )
)

# Persistent backup step
!cp {str(DATA_DIR / "alns_states_v1.pkl")} /kaggle/working/alns_states_v1.pkl
print("✓ ALNS intermediate states data backed up safely.")

Seed          : 1
Running ALNS on 500 instances to collect repair states...


Collected 314,843 rows  (pos_rate=0.292)
Saved to: /kaggle/working/bin-packing-optimization/bin_packing_optimization/hybrid_learning_metaheuristics/hybrid_alns/repair_model_training/training_data/alns_states_v1.pkl  (14.4 MB)
Next step: retrain with additional training data from /kaggle/working/bin-packing-optimization/bin_packing_optimization/hybrid_learning_metaheuristics/hybrid_alns/repair_model_training/training_data/alns_states_v1.pkl
✓ ALNS intermediate states data backed up safely.


## Step 4 — Generate supplementary synthetic data and retrain (v2)

Generates a smaller supplementary synthetic dataset (`synthetic_v2.pkl`, 2 000
instances) and retrains by merging it with the ALNS states collected in Step 3.

The v2 training enforces `require_alns_states=True`: it raises immediately if
no ALNS-tagged dataset is present, making the covariate-shift requirement
explicit and impossible to accidentally skip.

**Expected output**:
- Dataset summary with the ALNS ratio (ALNS rows / total rows) printed.
- Covariate-shift gate: `✓ ALNS states found: N rows (X.X% of total)`.
- Quality gate results for ROC-AUC and Average Precision.
- Model saved as `repair_model_v2.pkl` with the full provenance bundle.


In [16]:
# 4a. Generate supplementary dataset
generate_dataset(
    GenerateDatasetConfig(
        instances=2000,
        n_min=50,
        n_max=200,
        max_negatives=3,
        seed=SYNTHETIC_V2_SEED,
        workers=1,
        output=str(DATA_DIR / "synthetic_v2.pkl"),
    )
)

# 4b. Final optimization training
train_repair_model(
    TrainRepairModelConfig(
        data=[
            str(DATA_DIR / "synthetic_v2.pkl"),
            str(DATA_DIR / "alns_states_v1.pkl"),
        ],
        output=str(TRAINING_DIR / "repair_model_v2.pkl"),
        seed=SEED,
        min_roc_auc=0.80,
        min_average_precision=0.60,
        require_alns_states=True,
        cv_folds=3,
        no_learning_curves=True,
        no_plots=True,
    )
)

# Final persistent backup step
!cp {str(TRAINING_DIR / "repair_model_v2.pkl")} /kaggle/working/repair_model_v2.pkl
print("🎉 Pipeline run completely finished! Every checkpoint and file is now securely saved and visible inside your Kaggle Output explorer.")

DATASET GENERATION
  Instances          : 2000
  Instance size range: [50, 200] items
  Max negatives      : 3
  Seed               : 2
  Workers            : 1
  Output             : /kaggle/working/bin-packing-optimization/bin_packing_optimization/hybrid_learning_metaheuristics/hybrid_alns/repair_model_training/training_data/synthetic_v2.pkl


Generating dataset: 100%|██████████| 2000/2000 [00:04<00:00, 456.08it/s]



Dataset: 275,652 rows x 11 features
  Positive rate : 0.4117  (expected min: 0.2500)
  Class 0 (neg) : 162,154
  Class 1 (pos) : 113,498

Saved: /kaggle/working/bin-packing-optimization/bin_packing_optimization/hybrid_learning_metaheuristics/hybrid_alns/repair_model_training/training_data/synthetic_v2.pkl  (12.6 MB)
Next step: pass this file to train_repair_model.py via the data config field
REPRODUCIBILITY — SEEDS
  config.seed          : 42  (train/test split + model)
  Python random seed   : set to 42
  numpy random seed    : set to 42

PHASE 1: LOADING DATASET(S)

────────────────────────────────────────────────────────────
DATASET INTEGRITY CHECKS
────────────────────────────────────────────────────────────
  ✓ synthetic_v2.pkl
      source       : synthetic
      rows         : 275,652
      feature_ver  : 2
      pos_rate     : 0.4117
      NaN/inf      : none
      labels       : [0, 1]
  ✓ alns_states_v1.pkl
      source       : alns_states
      rows         : 314,843
      

## Step 5 — Optional benchmark

Run the Falkenauer-U benchmark to measure end-to-end solver quality with the
newly trained v2 model.  Set `RUN_BENCHMARK = True` to execute.

This step is intentionally disabled by default because it can take 10–30 minutes
depending on the runtime and instance size.


In [17]:
RUN_BENCHMARK = False

if RUN_BENCHMARK:
    from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns import (
        hybrid_alns_solver,
    )
    from bin_packing_optimization.utilities.benchmarking import create_benchmark

    benchmark = create_benchmark(
        dataset_key="falkenauer-u",
        solver_module=hybrid_alns_solver,
        time_limit=None,
    )
    benchmark.run(method=None, method_args={"max_iterations": 500})
    csv_path = benchmark.save_results_to_csv()
    print("Results saved to:", csv_path)

## Validation checklist

After running all cells, verify the following in the printed output:

- [ ] **Reproducibility**: seed constants printed at Step 3 setup match the values defined above
- [ ] **Integrity**: all dataset files pass `✓` checks (feature version, NaN, labels)
- [ ] **Dataset summary**: rows, positive rate, per-source counts printed for each training run
- [ ] **ALNS ratio**: v2 training reports `ALNS rows: N (X.X% of total)`
- [ ] **Covariate-shift gate**: `require_alns_states=True` accepted without error
- [ ] **Quality gates**: `✓ ROC-AUC ≥ 0.80` and `✓ Average Precision ≥ 0.60` for both models
- [ ] **Model saved**: `repair_model_v1.pkl` and `repair_model_v2.pkl` present in `TRAINING_DIR`
